# Introduction - Model Selection
In this section, we aim to identify the most suitable machine learning model for predicting housing prices in Melbourne.
We use the cleaned and preprocessed dataset and the previously developed feature engineering and preprocessing pipeline (melbourne_pipeline.pkl).
Multiple regression models are tested with cross-validation and hyperparameter tuning.
The final model is selected based on performance metrics and model interpretability.

In [ ]:
from statistics import variance

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor

from utils import FrequencyEncoder

## 1. Load dataset and pipeline

In [ ]:
full_pipeline = joblib.load("utils/melbourne_pipeline.pkl")
# Load cleaned dataset
df = pd.read_csv("data/melbourne_housing_cleaned_featEngr.csv")
df.head()

In [ ]:
X = df.drop(columns=["Price"])
y = np.log1p(df["Price"])  # target on log scale

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0
)

## 2. Define Models and Parameter Grids

Compare several regression models:
- Linear Regression (baseline)
- Lasso Regression
- Random Forest Regressor
- Gradient Boosting Regressor
- KNN 

Attach them to our pipeline dynamically.

In [ ]:
# ============================================
# 2. DEFINE MODELS AND PARAMETER GRIDS
# ============================================

models = {
    "LinearRegression": LinearRegression(),
    "Lasso": Lasso(max_iter=5000),
    "RandomForest": RandomForestRegressor(random_state=0, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(random_state=0),
    "KNN": KNeighborsRegressor(n_neighbors=5, weights='distance'),
}

param_grids = {
    "LinearRegression": {},  # no major hyperparameters
    "Lasso": {"model__alpha": [0.0001, 0.001, 0.01, 0.1, 1]},# Controls regularization strength
    "RandomForest": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [None, 10, 20],
        "model__min_samples_split": [2, 5]
    },
    "GradientBoosting": {
        "model__n_estimators": [100, 150],
        "model__learning_rate": [0.01, 0.05],
        "model__max_depth": [3]
    }, 
    "KNN": {
        "model__n_neighbors": list(range(3, 51, 2)),
        "model__weights": ["uniform", "distance"],
        "model__p": [1, 2]  # Manhattan vs Euclidean
    }
}

cv = KFold(n_splits=5, shuffle=True, random_state=0)
results = []

In [ ]:

# ============================================
# 3. MODEL SELECTION & GRID SEARCH
# ============================================

for name, model in models.items():
    print(f"\n=== Tuning {name} ===")
    pipe = joblib.load("utils/melbourne_pipeline.pkl")
    pipe.set_params(model=model)
    pipe.set_params(rfecv="passthrough")  # disable feature selection for fair comparison
    
    if model in ["RandomForest", "GradientBoosting"]:
        pipe.set_params(variance="passthrough")

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grids[name],
        scoring="r2",
        cv=cv,
        n_jobs=-1,
        verbose=1
    )
    grid.fit(X_train, y_train)

    results.append({
        "Model": name,
        "Best_Params": grid.best_params_,
        "Mean_R2": grid.best_score_,
        "Best_Estimator": grid.best_estimator_
    })

results_df = pd.DataFrame(results).sort_values("Mean_R2", ascending=False).reset_index(drop=True)
print("\n=== Cross-Validated Results ===")
display(results_df[["Model", "Mean_R2", "Best_Params"]])

In [ ]:
# ============================================
# 4. VISUALIZE MODEL PERFORMANCE
# ============================================

plt.figure(figsize=(8,5))
sns.barplot(data=results_df, x="Model", y="Mean_R2", palette="viridis", hue="Model",
            order=results_df.sort_values("Mean_R2", ascending=False)["Model"])
plt.title("Cross-Validated R² Comparison Across Models", fontsize=14)
plt.ylabel("Mean Cross-Validated R²")
plt.xlabel("")
plt.ylim(0, 1)
for index, row in results_df.iterrows():
    plt.text(index, row["Mean_R2"] + 0.01, f"{row['Mean_R2']:.3f}", ha='center', va='bottom', fontsize=9)
plt.show()

In [ ]:
# ============================================
# 5. FINAL MODEL EVALUATION
# ============================================
# Best model (from results_df)
best_model_name = results_df.iloc[0]["Model"]
best_pipe = results_df.iloc[0]["Best_Estimator"]

print(f" Best model selected: {best_model_name}")

# Evaluate on test set
best_pipe.fit(X_train, y_train)
y_pred = best_pipe.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(np.expm1(y_test), np.expm1(y_pred))
rmse = np.sqrt(mean_squared_error(np.expm1(y_test), np.expm1(y_pred)))

print(f"R² (log scale): {r2:.4f}")
print(f"MAE (original scale): {mae:,.0f}")
print(f"RMSE (original scale): {rmse:,.0f}")

In [ ]:
# ============================================
# 6. PREDICTED VS ACTUAL
# ============================================

plt.figure(figsize=(6,6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
plt.title(f"Predicted vs Actual (log Price) — {best_model_name}")
plt.xlabel("Actual (log Price)")
plt.ylabel("Predicted (log Price)")
plt.show()

plt.figure(figsize=(6,6))
sns.scatterplot(x=np.expm1(y_test), y=np.expm1(y_pred), alpha=0.6)
plt.plot([np.expm1(y_test).min(), np.expm1(y_test).max()],
         [np.expm1(y_test).min(), np.expm1(y_test).max()], "r--")
plt.title(f"Predicted vs Actual (Original Price) — {best_model_name}")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.show()
